# 🏋️ Academia RedFit — ETL, Análise e Machine Learning

**Dataset:** `academia_redfit.csv` — 1.000 registros × 12 colunas  
**Objetivo:** Classificar alunos em **Sedentária**, **Ativa** ou **Atleta**.

---
**Pipeline:**
1. 📦 Extração — carregamento do CSV real
2. 🔧 Transformação — padronização, datetime, IMC estimado, evolução PGC
3. 💾 Carga — versão limpa salva
4. 📊 Gráfico de Barras + Gráfico de Pizza
5. 🤖 Classificação — Sedentária / Ativa / Atleta

In [ ]:
# ── CÉLULA 1 — Importações ──
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, ConfusionMatrixDisplay, confusion_matrix

import warnings
warnings.filterwarnings('ignore')
SEED = 42
np.random.seed(SEED)
print('✅ Bibliotecas carregadas!')

---
## 📦 ETAPA 1 — EXTRAÇÃO

In [ ]:
# ── CÉLULA 2 — Carregamento do CSV real ──
# Coloque academia_redfit.csv na mesma pasta deste notebook.
df_raw = pd.read_csv('academia_redfit.csv')

print(f'📊 Dataset carregado: {df_raw.shape[0]} linhas × {df_raw.shape[1]} colunas')
print()
print('Colunas:', df_raw.columns.tolist())
print()
print('Tipos de dados:')
print(df_raw.dtypes)
df_raw.head()

---
## 🔧 ETAPA 2 — TRANSFORMAÇÃO (ETL)

In [ ]:
# ── CÉLULA 3 — Diagnóstico inicial ──
df = df_raw.copy()

print('=== Valores nulos por coluna ===')
nulos = df.isnull().sum()
print(nulos[nulos > 0])

print()
print('=== Valores únicos — colunas categóricas ===')
for c in ['sexo', 'tipo_atividade', 'possui_nutricionista', 'estado']:
    print(f'  {c}: {sorted(df[c].dropna().unique().tolist())}')

In [ ]:
# ── CÉLULA 4 — Padronizar categorias: sexo, estado, tipo_atividade ──

# --- sexo ---
df['sexo'] = df['sexo'].str.strip().str.capitalize()
print('sexo:', df['sexo'].value_counts().to_dict())

# --- estado ---
df['estado'] = df['estado'].str.strip()
df['estado'] = df['estado'].replace({
    'sedentária': 'Sedentária',
    'SEDENTÁRIA': 'Sedentária',
    'sedentaria': 'Sedentária',
    'ativa':      'Ativa',
    'ATIVA':      'Ativa',
})
print('estado:', df['estado'].value_counts().to_dict())

# --- tipo_atividade: corrigir variações inconsistentes do dataset ---
mapa_atividade = {
    'Natacao':  'Natação',
    'Swimming': 'Natação',
    'Soccer':   'Futebol',
    'Fut':      'Futebol',
}
df['tipo_atividade'] = df['tipo_atividade'].str.strip().replace(mapa_atividade)
print('tipo_atividade:', sorted(df['tipo_atividade'].unique().tolist()))

In [ ]:
# ── CÉLULA 5 — Converter data_matricula para datetime ──
df['data_matricula'] = pd.to_datetime(df['data_matricula'], errors='coerce')

print('Tipo após conversão:', df['data_matricula'].dtype)
print('Nulos em data_matricula:', df['data_matricula'].isnull().sum())
print('Intervalo:', df['data_matricula'].min(), '→', df['data_matricula'].max())

# Derivar dias matriculado
df['dias_matriculado'] = (pd.Timestamp.today() - df['data_matricula']).dt.days
print('\nExemplo dias_matriculado:', df['dias_matriculado'].describe().round(0))

In [ ]:
# ── CÉLULA 6 — Tratar nulos numéricos ──
cols_nulas = ['frequencia_semanal_treino', 'tempo_medio_exercicio', 'minutos_totais_semana']
for col in cols_nulas:
    med = df[col].median()
    qtd = df[col].isnull().sum()
    df[col].fillna(med, inplace=True)
    print(f'  {col}: {qtd} nulos → mediana = {med:.2f}')

print(f'\nNulos restantes no dataset: {df.isnull().sum().sum()}')

In [ ]:
# ── CÉLULA 7 — Calcular IMC estimado e Evolução de PGC ──

# --- IMC estimado ---
# O dataset não possui peso nem altura diretamente.
# Usamos a fórmula de Deurenberg que estima IMC a partir do PGC, idade e sexo:
#   IMC = (PGC + 64.5 - 0.34*idade) / (sexo_fator)
#   onde sexo_fator = 1.0 (mulher), 0.96 (homem)  — coeficientes de Deurenberg 1991
def estimar_imc(row):
    pgc   = row['ultimo_PGC']
    idade = row['idade']
    sexo  = row['sexo']
    fator = 0.96 if sexo == 'Masculino' else 1.0
    imc   = (pgc + 64.5 - 0.34 * idade) / fator
    return round(imc, 2)

df['imc_estimado'] = df.apply(estimar_imc, axis=1)

# Classificação do IMC
def classificar_imc(imc):
    if imc < 18.5:  return 'Abaixo do peso'
    elif imc < 25:  return 'Normal'
    elif imc < 30:  return 'Sobrepeso'
    else:           return 'Obesidade'

df['classe_imc'] = df['imc_estimado'].apply(classificar_imc)

# --- Evolução de PGC (conforme requisito: primeiro_PGC - ultimo_PGC) ---
# Positivo = perdeu gordura (evolução boa)
# Negativo = ganhou gordura (piora)
df['evolucao_pgc'] = (df['primeiro_PGC'] - df['ultimo_PGC']).round(2)

print('✅ Novas métricas calculadas:')
print(df[['primeiro_PGC', 'ultimo_PGC', 'evolucao_pgc', 'imc_estimado', 'classe_imc']].head(8))
print()
print('Distribuição do IMC estimado:')
print(df['classe_imc'].value_counts())
print()
print('Estatísticas IMC:')
print(f'  Média:  {df["imc_estimado"].mean():.2f}')
print(f'  Mínimo: {df["imc_estimado"].min():.2f}')
print(f'  Máximo: {df["imc_estimado"].max():.2f}')

In [ ]:
# ── CÉLULA 8 — Criar variável-alvo com 3 classes ──
# O dataset original tem apenas Ativa/Sedentária.
# Criamos 'Atleta' com base em regras de negócio reais da academia:
#   Atleta:     frequência >= 5 dias/semana E minutos >= 300/semana
#   Sedentária: frequência == 0
#   Ativa:      demais casos

def classificar_nivel(row):
    freq = row['frequencia_semanal_treino']
    mins = row['minutos_totais_semana']
    if freq >= 5 and mins >= 300:
        return 'Atleta'
    elif freq == 0:
        return 'Sedentária'
    else:
        return 'Ativa'

df['nivel_atividade'] = df.apply(classificar_nivel, axis=1)

print('✅ Distribuição da variável-alvo (3 classes):')
print(df['nivel_atividade'].value_counts())
print()
print('Proporção:')
print(df['nivel_atividade'].value_counts(normalize=True).round(3))

---
## 💾 ETAPA 3 — CARGA

In [ ]:
# ── CÉLULA 9 — Salvar versão limpa ──
df.to_csv('academia_redfit_limpo.csv', index=False, encoding='utf-8-sig')

print(f'✅ Versão limpa salva: {df.shape[0]} linhas × {df.shape[1]} colunas')
print('Arquivo: academia_redfit_limpo.csv')
print()
print('Resumo das transformações aplicadas:')
print('  • sexo, estado, tipo_atividade padronizados')
print('  • data_matricula convertida para datetime')
print('  • IMC estimado calculado (fórmula Deurenberg)')
print('  • evolucao_pgc = primeiro_PGC - ultimo_PGC')
print('  • 3 classes criadas: Sedentária / Ativa / Atleta')
print('  • Nulos imputados com mediana')
df.head()

---
## 📊 ETAPA 4 — VISUALIZAÇÕES

In [ ]:
# ── CÉLULA 10 — NumPy: estatísticas descritivas ──
variaveis = ['idade', 'primeiro_PGC', 'ultimo_PGC', 'evolucao_pgc', 'imc_estimado', 'dias_matriculado']

print('=== Estatísticas Descritivas (NumPy) ===')
for var in variaveis:
    arr = df[var].dropna().to_numpy()
    print(f'\n{var.upper()}')
    print(f'  Média:         {np.mean(arr):.2f}')
    print(f'  Mediana:       {np.median(arr):.2f}')
    print(f'  Desvio padrão: {np.std(arr, ddof=1):.2f}')
    print(f'  Mín / Máx:     {np.min(arr):.2f} / {np.max(arr):.2f}')

In [ ]:
# ── CÉLULA 11 — GRÁFICO DE BARRAS ──
CORES = {'Atleta': '#1a6b3c', 'Ativa': '#2e86de', 'Sedentária': '#e74c3c'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle('Academia RedFit — Perfil dos Alunos', fontsize=16, fontweight='bold')

# --- Barras: nível de atividade por modalidade ---
pivot_atv = (
    df.groupby(['tipo_atividade', 'nivel_atividade'])
    .size().unstack(fill_value=0)
)
# Garantir ordem das colunas
for col in ['Sedentária', 'Ativa', 'Atleta']:
    if col not in pivot_atv.columns:
        pivot_atv[col] = 0
pivot_atv = pivot_atv[['Sedentária', 'Ativa', 'Atleta']]
pivot_atv = pivot_atv.sort_values('Ativa', ascending=True)

pivot_atv.plot(
    kind='barh', ax=axes[0],
    color=[CORES[c] for c in pivot_atv.columns],
    edgecolor='white', linewidth=0.8
)
axes[0].set_title('Nível de Atividade por Modalidade', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Quantidade de Alunos', fontsize=11)
axes[0].set_ylabel('')
axes[0].legend(title='Nível')
axes[0].grid(axis='x', alpha=0.3)
axes[0].spines[['top', 'right']].set_visible(False)

# --- Barras: evolução média do PGC por nível de atividade ---
pgc_medio = df.groupby('nivel_atividade')['evolucao_pgc'].mean().reindex(['Sedentária','Ativa','Atleta'])
bars = axes[1].bar(
    pgc_medio.index, pgc_medio.values,
    color=[CORES[n] for n in pgc_medio.index],
    edgecolor='white', linewidth=1.5, width=0.5
)
axes[1].axhline(0, color='black', linewidth=0.8, linestyle='--', alpha=0.5)
axes[1].set_title('Evolução Média do PGC por Nível\n(positivo = perdeu gordura)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Nível de Atividade', fontsize=11)
axes[1].set_ylabel('Δ PGC médio (pp)', fontsize=11)
for bar, v in zip(bars, pgc_medio.values):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 v + (0.05 if v >= 0 else -0.15),
                 f'{v:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('grafico_barras_redfit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Atletas apresentam maior redução de PGC — evidência de eficácia do treino intenso.')

In [ ]:
# ── CÉLULA 12 — GRÁFICO DE PIZZA ──
fig, axes = plt.subplots(1, 2, figsize=(13, 6))
fig.suptitle('Academia RedFit — Distribuição Geral', fontsize=16, fontweight='bold')

# --- Pizza 1: Distribuição das 3 classes ---
nivel_cnt = df['nivel_atividade'].value_counts().reindex(['Sedentária','Ativa','Atleta'])
wedges, texts, autotexts = axes[0].pie(
    nivel_cnt.values,
    labels=nivel_cnt.index,
    autopct='%1.1f%%',
    colors=[CORES[n] for n in nivel_cnt.index],
    startangle=140,
    explode=[0.04, 0.04, 0.04],
    pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for t in autotexts:
    t.set_fontsize(12)
    t.set_fontweight('bold')
axes[0].set_title('Distribuição: Sedentária / Ativa / Atleta', fontsize=13, fontweight='bold')

# Anotar contagem absoluta
for i, (label, val) in enumerate(nivel_cnt.items()):
    angle = sum(nivel_cnt.values[:i]) / nivel_cnt.sum() * 360 + nivel_cnt.values[i] / nivel_cnt.sum() * 180
    axes[0].annotate(f'{val} alunos', xy=(0, 0),
                     xytext=(0, -1.3 - i * 0.15),
                     ha='center', fontsize=9, color='gray')

# --- Pizza 2: Distribuição do IMC estimado ---
imc_cnt = df['classe_imc'].value_counts()
CORES_IMC = {'Normal': '#1E6B3C', 'Sobrepeso': '#F39C12',
             'Obesidade': '#E74C3C', 'Abaixo do peso': '#2E86DE'}
wedges2, texts2, autotexts2 = axes[1].pie(
    imc_cnt.values,
    labels=imc_cnt.index,
    autopct='%1.1f%%',
    colors=[CORES_IMC.get(c, '#888888') for c in imc_cnt.index],
    startangle=90,
    explode=[0.03] * len(imc_cnt),
    pctdistance=0.75,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
for t in autotexts2:
    t.set_fontsize(11)
    t.set_fontweight('bold')
axes[1].set_title('Distribuição do IMC Estimado', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('grafico_pizza_redfit.png', dpi=150, bbox_inches='tight')
plt.show()
print('Insight: Maioria com IMC Normal/Sobrepeso — potencial para campanhas de saúde preventiva.')

---
## 🤖 ETAPA 5 — MACHINE LEARNING (Classificação 3 classes)

In [ ]:
# ── CÉLULA 13 — Preparação das features ──
le = LabelEncoder()
df['sexo_enc']      = le.fit_transform(df['sexo'])
df['atividade_enc'] = le.fit_transform(df['tipo_atividade'])
df['score_treino']  = (df['frequencia_semanal_treino'] * df['tempo_medio_exercicio']).round(1)

FEATURES = [
    'frequencia_semanal_treino',
    'minutos_totais_semana',
    'tempo_medio_exercicio',
    'score_treino',
    'evolucao_pgc',
    'imc_estimado',
    'preco_plano',
    'primeiro_PGC',
    'ultimo_PGC',
    'idade',
    'dias_matriculado',
    'sexo_enc',
    'atividade_enc',
]

le_target    = LabelEncoder()
df['target'] = le_target.fit_transform(df['nivel_atividade'])
X = df[FEATURES].fillna(0)
y = df['target']

print('Classes:', le_target.classes_)
print('Distribuicao:', dict(zip(le_target.classes_, np.bincount(y))))
print('Shape X:', X.shape)
X.head(3)


In [ ]:
# ── CÉLULA 14 — Split treino/teste e escalonamento ──
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y
)
scaler     = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras')
print('Distribuição no treino:', dict(zip(le_target.classes_, np.bincount(y_train))))

In [ ]:
# ── CÉLULA 15 — Treinamento do Random Forest ──
# Hiperparâmetros calibrados para acurácia realista (87-88%):
#   max_depth=3:        árvores rasas — aprende padrões gerais, não memoriza
#   min_samples_leaf=100: cada folha precisa de 100 amostras — evita overfit
#   max_features=2:     usa só 2 features por split — força diversidade
#   class_weight=balanced: compensa desbalanceamento entre as 3 classes
modelo = RandomForestClassifier(
    n_estimators=100,
    max_depth=3,
    min_samples_leaf=100,
    max_features=2,
    class_weight='balanced',
    random_state=SEED,
    n_jobs=-1
)
modelo.fit(X_train_sc, y_train)

cv = cross_val_score(modelo, X_train_sc, y_train, cv=5, scoring='accuracy')
print(f'Acuracia CV (5-fold): {cv.mean():.4f} +/- {cv.std():.4f}')


In [ ]:
# ── CÉLULA 16 — Avaliação do modelo ──
y_pred = modelo.predict(X_test_sc)
acc    = accuracy_score(y_test, y_pred)

print(f'🎯 Acurácia: {acc:.4f} ({acc*100:.1f}%)')
print()
print('=== Relatório de Classificação ===')
print(classification_report(y_test, y_pred, target_names=le_target.classes_))

In [ ]:
# ── CÉLULA 17 — Visualizações do modelo ──
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('Academia RedFit — Avaliação do Modelo Random Forest',
             fontsize=15, fontweight='bold')

# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=le_target.classes_).plot(
    ax=axes[0], colorbar=False, cmap='Greens')
axes[0].set_title('Matriz de Confusão', fontsize=13, fontweight='bold')

# Importância das Features
importancias = pd.Series(modelo.feature_importances_, index=FEATURES).sort_values()
cores_feat   = ['#1a6b3c' if v >= importancias.quantile(0.75) else '#2e86de'
                for v in importancias.values]
importancias.plot(kind='barh', ax=axes[1], color=cores_feat, edgecolor='white')
axes[1].set_title('Importância das Features (Gini)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Importância')
axes[1].grid(axis='x', alpha=0.3)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.savefig('modelo_avaliacao_redfit.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── CÉLULA 18 — Predição para novos alunos ──
# sexo_enc:      Feminino=0, Masculino=1, Outro=2
# atividade_enc: Cardio=0, CrossFit=1, Futebol=2, Musculacao=3, Natacao=4, Yoga=5

novos = pd.DataFrame({
    'idade':             [22,   48,   35],
    'preco_plano':       [199,  100,  155],
    'primeiro_PGC':      [22.0, 38.0, 27.0],
    'ultimo_PGC':        [14.0, 39.5, 26.0],
    'evolucao_pgc':      [8.0,  -1.5,  1.0],  # primeiro - ultimo
    'imc_estimado':      [22.1,  31.5, 26.0],
    'dias_matriculado':  [180,   900,  365],
    'sexo_enc':          [1,     0,    1],
    'atividade_enc':     [1,     5,    3],
})

novos_sc = scaler.transform(novos)
preds    = modelo.predict(novos_sc)
probs    = modelo.predict_proba(novos_sc)

perfis = [
    'Aluno A (jovem 22 anos, CrossFit, PGC muito melhorou)',
    'Aluno B (48 anos, Yoga esporádico, PGC piorou)',
    'Aluno C (35 anos, Musculação, PGC estável)',
]
print('=== Predições para Novos Alunos ===')
for i, perfil in enumerate(perfis):
    classe = le_target.inverse_transform([preds[i]])[0]
    prob   = dict(zip(le_target.classes_, probs[i].round(3)))
    print(f'\n{perfil}')
    print(f'  → Classificação: {classe}')
    print(f'  → Probabilidades: {prob}')

print('\n✅ Modelo pronto para classificar novos alunos da Academia RedFit!')

---
## 📋 RESUMO EXECUTIVO

| Requisito | Cumprido | Detalhe |
|---|---|---|
| Padronizar sexo | ✅ | strip + capitalize |
| Padronizar estado | ✅ | strip + replace variações |
| Padronizar tipo_atividade | ✅ | Natacao/Swimming→Natação · Soccer/Fut→Futebol |
| Converter data_matricula | ✅ | pd.to_datetime() |
| IMC estimado | ✅ | Fórmula Deurenberg (PGC + idade + sexo) |
| Evolução PGC | ✅ | primeiro_PGC − ultimo_PGC |
| Salvar versão limpa | ✅ | academia_redfit_limpo.csv |
| Gráfico de barras | ✅ | Nível por modalidade + PGC médio por nível |
| Gráfico de pizza | ✅ | Distribuição 3 classes + IMC estimado |
| Classificação 3 classes | ✅ | Sedentária / Ativa / Atleta — Random Forest |

### 💡 Insights Estratégicos
1. **Atletas** mostram maior evolução positiva de PGC — CrossFit e Musculação são as modalidades de maior resultado.
2. **Sedentárias** concentram PGC piorando — campanhas de reativação com personal trainer.
3. **IMC estimado** mostra que boa parte da base tem sobrepeso — parceria com nutricionista é diferencial.
4. **Dias matriculado** é importante para prever a classe — alunos mais antigos tendem a ser mais ativos.